## Exporters written in python to generate JSON file for connection with WP1

In [1]:
#2024-10

# See "Example Notebooks Streamlined API Example Notebooks Get Attributes" and
# "Example Notebooks Streamlined API Example Notebooks Layouts" from the Granta MI STK documentation

# Import the required libraries
from GRANTA_MIScriptingToolkit import granta as mpy
import pandas as pd
from datetime import datetime
import json

usernameVal = 'STK_user'
passwordVal = 'P455w07d!'

session = mpy.connect('https://nanomecommons.grantami.com/mi_servicelayer', user_name=usernameVal, password=passwordVal)

In [2]:
# Connect to the database
dbKey = "MI_Blank_v2021"
tableName = "Indentation test data"
layoutName = "Indentation test data"
recordName = "SiNx CSM High-Speed"

db = session.get_db(db_key=dbKey)
tab = db.get_table(tableName)
indentation_layout = tab.layouts[layoutName]
indentation_layout.categories
attributes_by_category = indentation_layout.attributes_by_category
record = tab.search_for_records_by_name(recordName)[0]

data = []
for category, attributes in attributes_by_category.items():
    for attribute in attributes:
        data.append([category, attribute])

df = pd.DataFrame(data, columns=['Categories', 'Attributes'])
print(df)


              Categories                                         Attributes
0    General information                                          Test Date
1    General information                                           Operator
2    General information                                            Partner
3   Specimen information                                         Grade name
4   Specimen information                                      Specimen code
5   Specimen information                                    Material Family
6          Test settings                                  Test equipment ID
7          Test settings               Type of Indentation Test Measurement
8          Test settings                                   Temperature Test
9          Test settings                          Indentation Controle Mode
10         Test settings                                      Indenter type
11         Test settings                                      Indenter name
12         T

In [3]:
# Print the list of all attributes in the table
print('{:30.30} - {:^10.10} - {:^10.10} - {:^10.10} - {:^20.20} - {:^20.20}'.format('Name', 'Type', 'Unit', 'Meta?', 'Meta-attributes', 'Column headers'))
print('-'*115)
for name, att in tab.attributes.items():
    if att.type == 'FUNC':
        func = record.attributes[name]
        output = '{:30.30} - {:^10.10} - {:^10.10} - {:^10.10} - {:^20.20} - {:^20.20}'.format(name, att.type, att.unit, str(att.is_meta), str(att.meta_attributes), str(func.column_headers))
        print(output)
    elif att.type == 'TABL':
        hist = record.attributes[name]
        output = '{:30.30} - {:^10.10} - {:^10.10} - {:^10.10} - {:^20.20} - {:^20.20}'.format(name, att.type, att.unit, str(att.is_meta), str(att.meta_attributes), str(hist.columns))
        print(output)
    else:
        output = '{:30.30} - {:^10.10} - {:^10.10} - {:^10.10} - {:^20.20}'.format(name, att.type, att.unit, str(att.is_meta), str(att.meta_attributes))
        print(output)

Name                           -    Type    -    Unit    -   Meta?    -   Meta-attributes    -    Column headers   
-------------------------------------------------------------------------------------------------------------------
Contact depth (mean)           -    POIN    -     nm     -   False    - {'Maximum': <Attribu
Peak load (mean)               -    POIN    -     N      -   False    - {'Maximum': <Attribu
Stiffness (mean)               -    POIN    -    N/nm    -   False    - {'Maximum': <Attribu
Contact area (mean)            -    POIN    -    nm^2    -   False    - {'Maximum': <Attribu
Reduced elastic modulus (mean) -    POIN    -    GPa     -   False    - {'Maximum': <Attribu
Hardness (mean)                -    POIN    -    GPa     -   False    - {'Maximum': <Attribu
Test equipment ID              -    STXT    -            -   False    -          {}         
Manufacturer                   -    STXT    -            -   False    -          {}         
Model                   

In [4]:
# Create a json file listing attributes for a given data table from Granta MI

now = datetime.now()
formatted_now = now.strftime("%Y%m%d")
jsonName = f"{tableName}_attributes_{formatted_now}.json"

# Collect detailed attribute information
detailed_attributes = {}
for category, attributes in attributes_by_category.items():
    detailed_attributes[category] = []
    for attribute in attributes:
        if attribute in tab.attributes:
            att = tab.attributes[attribute]
            meta_attributes_info = []
            for meta_key, meta_attr in att.meta_attributes.items():
                meta_info = {
                    'MetaName': meta_attr.name,
                    'MetaType': meta_attr.type,
                    'MetaUnit': meta_attr.unit,
                    'IsMeta': 'True'
                }
                if meta_attr.type == 'TABL':
                    hist = record.attributes[attribute].meta_attributes[meta_attr.name]
                    units_list = hist.units.data[0] if hist.units.data else None
                    meta_info['ColumnHeaders'] = [{'Name': hist.columns, 'Unit': units_list}]
                elif meta_attr.type == 'FUNC':
                    func = record.attributes[attribute].meta_attributes[meta_attr.name]
                    meta_info['ColumnHeaders'] = [{'Name': func.column_headers}]
                meta_attributes_info.append(meta_info)
            detailed_info = {
                'Name': attribute,
                'Type': att.type,
                'Unit': att.unit,
                'IsMeta': att.is_meta
            }

            if att.type == 'TABL':
                hist = record.attributes[attribute]
                units_list = hist.units.data[0] if hist.units.data else None
                detailed_info['ColumnHeaders'] = [{'Name': hist.columns, 'Unit': units_list}]
            elif att.type == 'FUNC':
                func = record.attributes[attribute]
                detailed_info['ColumnHeaders'] = [{'Name': func.column_headers}]
            detailed_attributes[category].append(detailed_info)
            detailed_attributes[category].append(meta_attributes_info)

# Export the list of outputs to a JSON file
with open(jsonName, 'w') as json_file:
    json.dump(detailed_attributes, json_file, indent=4)